In [1]:
import os
import math
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn.functional as F  
from torch.utils.tensorboard import SummaryWriter

from Utils.CADTensorGenerator import CADTensorGenerator
from Utils.CADVisualizer   import CADVisualizer
from HDVClassNet.PP_net import PPNet
from HDVClassNet.VoronoiDecorder import VoronoiDecoder
from Training.MainTrain import TrainingConfig, NN_Trainer
from neuraltomo_fem import run_fem_loss
from problems.ThickenShell import ThickenShell

import pyvista as pv
try:
    pv.set_jupyter_backend("trame")
except Exception:
    pass

# ---- Reproducibility (recommended for D_params comparisons) ----
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
print("Code Directory:", BASE)
TesPartsDir = BASE / "Testparts" 
print("Test Step files Directory:", TesPartsDir)
# ---- Device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


Code Directory: /home/arash/VoronoiImp-main
Test Step files Directory: /home/arash/VoronoiImp-main/Testparts
device: cuda


In [ ]:
viz = CADVisualizer()
# Laoding model and extracting mesh and tensors as input
FreeFormSurf1  = TesPartsDir / "FreeFormCrv1.stp"
FreeFormSurf2A = TesPartsDir / "FreeFormSurf2A.STEP"
YachtBodypart  = TesPartsDir / "YachtBodypart.stp"
CircularSurf1  = TesPartsDir / "CircularSurf1.stp"
Cube           = TesPartsDir / "Cube.stp"
CircularSur2   = TesPartsDir / "CircularSur2.stp"
Conic          = TesPartsDir / "Conic.stp"
CircularHoles  = TesPartsDir / "CircularHoles.stp"
FullCylinder   = TesPartsDir / "FullCylinder.stp"
Sphere         = TesPartsDir / "Sphere.stp"
SphereTap      = TesPartsDir / "SphereTap.stp"
Tidebottle     = TesPartsDir / "Tidebottle.STEP"


shape_path = FullCylinder

Case_name = shape_path.stem
print(Case_name)
generator = CADTensorGenerator(
    deflection=0.1,
    angle=0.001,
    metric_tol=1e-9,
    det_min=1e-5,
    n_u=75,
    n_v=75,
    device=device,
)

mesh_df, faces_df, tensors = generator.generate_from_file(
    shape_path=str(shape_path),
    input_ring=1,
    mode="mesh", #"1: mesh" "2:Sampled_points "
    M_per_face=2000,
    pool_size_factor=10,
    fps_pool_factor=4,
    use_fps=True,
    triangulation_max_edge_rel=0.1,
)

uv = tensors["uv"]
points_xyz = tensors["points_xyz"]
uv = tensors["uv"]
Xu = tensors["Xu"]
Xv = tensors["Xv"]
points_xyz = tensors["points_xyz"]
face_areas = tensors["face_areas"]
faces_ijk = tensors["faces_ijk"]
face_id = tensors["face_id"]
boundary_idx_ring1 = tensors["boundary_idx_ring1"]
pv_faces = tensors["pv_faces"]
bbx_all = list(tensors["BBX"].values())



xmin = min(b["xmin"] for b in bbx_all)
xmax = max(b["xmax"] for b in bbx_all)
ymin = min(b["ymin"] for b in bbx_all)
ymax = max(b["ymax"] for b in bbx_all)
zmin = min(b["zmin"] for b in bbx_all)
zmax = max(b["zmax"] for b in bbx_all)

dx = xmax - xmin
dy = ymax - ymin
dz = zmax - zmin


print(f"Number of faces: {tensors['num_faces']}")
print(f"Number of Sampled points: {uv.shape[0]}")
print(f"Global BBX dimensions: dx={dx:.4f}, dy={dy:.4f}, dz={dz:.4f}")

viz.visualize_show_Model(points_xyz, pv_faces)


pts = points_xyz.detach().cpu().numpy()
cloud = pv.PolyData(pts)
plotter = pv.Plotter()
plotter.add_mesh(cloud, render_points_as_spheres=True, point_size=6)
plotter.show()

FullCylinder
MinVolFrac: 0.03999999910593033
Number of faces: 1
Number of Sampled points: 5700
Global BBX dimensions: dx=81.9640, dy=81.9820, dz=100.0000


Widget(value='<iframe src="http://localhost:46485/index.html?ui=P_0x7346e0526a70_0&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:46485/index.html?ui=P_0x7346e0526a10_1&reconnect=auto" class="pyvi…

Exception raised
KeyError('0a5e97f2dd39a85e3652b87520a028ba_9366f')
Traceback (most recent call last):
  File "/home/arash/miniconda3/envs/VD_Imp_UB/lib/python3.10/site-packages/wslink/protocol.py", line 312, in onCompleteMessage
    results = func(*args, **kwargs)
  File "/home/arash/miniconda3/envs/VD_Imp_UB/lib/python3.10/site-packages/trame_vtk/modules/vtk/protocols/local_rendering.py", line 33, in get_array
    self.context.get_cached_data_array(data_hash, binary)
  File "/home/arash/miniconda3/envs/VD_Imp_UB/lib/python3.10/site-packages/trame_vtk/modules/vtk/serializers/synchronization_context.py", line 41, in get_cached_data_array
    cache_obj = self.data_array_cache[p_md5]
KeyError: '0a5e97f2dd39a85e3652b87520a028ba_9366f'

Exception raised
KeyError('15e6a16fe206d9a5a876355aab1687f9_21764L')
Traceback (most recent call last):
  File "/home/arash/miniconda3/envs/VD_Imp_UB/lib/python3.10/site-packages/wslink/protocol.py", line 312, in onCompleteMessage
    results = func(*args, 

In [3]:
fixed_height_shell= 0.5
# shell_problem = ThickenShell(
#     thickness=fixed_height_shell,
#     BC_dir = "x",
#     Load_magnitude=0.0001,
#     voxel_size=0.2,
#     extra_layers=1,
#     tensors=tensors,
#     tangential_tol=0.1,
# )
shell_problem =None
fem =None
#fem = run_fem_loss.NeuralTOMOFEM(shell_problem, device=device, isotropic=False)
# shell_problem.debug_voxel_stats()
# shell_problem.show_voxels_surface_and_bc()

In [8]:
cfg = TrainingConfig(
    seed_number=10,
    use_anisotropy=False,
    fixed_height=fixed_height_shell,
    target_volfrac=0.1,
    seed_repulsion_sigma=0.08,
    boundary_margin=0.05,
    w_min=0.0005,
    w_max=0.5,
    freeze_w = False,

    lam_fem=0.0,
    lam_vol=20.0,
    lam_rep=0.015,
    lam_bnd=0.002,

    lam_strut=0.0002,
    lam_strut_edge=1.0,
    lam_strut_void=0.2,

    lr_seed_refine=0.003,
    lr_delta_head=1e-4,
    lr_mlp=2e-4,
    lr_w_head=2e-3,
    lr_h_head=2e-4,

    comp_normalize_by=None,
    normalize_losses=False,

    fem_density_floor=0.02,
    skip_bad_fem_steps=True,

    num_steps=10000,
    context_vector_size=8,

    tau=0.1,
    beta=0.02,

    log_every=100,
    early_stop_start=500,
    patience=500,
    min_delta=1e-8,

    experiment_name=str(Case_name),
    tensorboard_enabled=True,
    tb_log_histograms_every=100,

    # new fields for the rewritten trainer
    scheduler_milestones=(500, 1000),
    scheduler_gamma=0.2,
    Offset_scale=5,
    save_fem_debug_history=True,
    grad_clip_norm=1.0,
)

trainer = NN_Trainer(
    generator=generator,
    viz=viz,
    decoder_cls=VoronoiDecoder,
    ppnet_cls=PPNet,
    fem=fem,
    shell_problem=shell_problem,
    config=cfg,
)

result = trainer.train(
    face_tensors=tensors["face_tensors"],
)

trainer.visualize_result_stepwise(result, points_xyz, faces_ijk)
trainer.visualize_result_final(result, points_xyz, faces_ijk, thr=0.5, show_solid=False)

TensorBoard log dir: runs/FullCylinder
New best_step=0 | best_score=0.659544 | vol_eff=0.281468 | comp=0.000000e+00 | w=2.518401e-01
[00000] L_total=6.5954e-01 | L_vol=3.293e-02 L_fem=0.000e+00 L_strut=7.461e-02 L_rep=5.912e-05 L_bnd=4.594e-01 | vol=0.288 vol_eff=0.281 (/0.100) comp=0.000e+00 w=2.518e-01 | Lse=7.461e-02 Lsv=0.000e+00 rho(min/mean/max)=0.001/0.290/0.874 | rho_b=0.098 rho_v=0.260 | Δrho=0.00e+00 Δseed=0.00e+00 grad_mean=6.91e-03 | fem=OK | best=6.5954e-01@0
[00100] L_total=4.5381e-01 | L_vol=2.266e-02 L_fem=0.000e+00 L_strut=1.235e-01 L_rep=9.493e-04 L_bnd=3.119e-01 | vol=0.259 vol_eff=0.251 (/0.100) comp=0.000e+00 w=8.364e-02 | Lse=1.217e-01 Lsv=8.900e-03 rho(min/mean/max)=0.000/0.261/0.889 | rho_b=0.098 rho_v=0.232 | Δrho=2.27e-01 Δseed=5.48e-02 grad_mean=1.26e-02 | fem=OK | best=3.8254e-01@97
[00200] L_total=7.2317e-04 | L_vol=3.280e-07 L_fem=0.000e+00 L_strut=2.988e-02 L_rep=1.232e-03 L_bnd=3.461e-01 | vol=0.115 vol_eff=0.101 (/0.100) comp=0.000e+00 w=5.845e-03 | Lse

Widget(value='<iframe src="http://localhost:46485/index.html?ui=P_0x7342380a70d0_11&reconnect=auto" class="pyv…

threshold=0.5 (manual) | solid%=20.596%


Widget(value='<iframe src="http://localhost:46485/index.html?ui=P_0x7344615d9ba0_12&reconnect=auto" class="pyv…

(UnstructuredGrid (0x7341f5c7bfa0)
   N Cells:    3015
   N Points:   1971
   X Bounds:   -4.096e+01, 4.100e+01
   Y Bounds:   -4.099e+01, 4.099e+01
   Z Bounds:   0.000e+00, 1.000e+02
   N Arrays:   3,
 0.5)

In [5]:
%load_ext tensorboard
%tensorboard --logdir runs

In [6]:
for t in [0.3,0.4,0.5, 0.6, 0.7]:
    trainer.visualize_result_final(result, points_xyz, faces_ijk, thr=t, show_solid=True)

threshold=0.3 (manual) | solid%=42.965%


Widget(value='<iframe src="http://localhost:46485/index.html?ui=P_0x73446309ab30_4&reconnect=auto" class="pyvi…

threshold=0.4 (manual) | solid%=36.807%


Widget(value='<iframe src="http://localhost:46485/index.html?ui=P_0x73446309a7d0_5&reconnect=auto" class="pyvi…

threshold=0.5 (manual) | solid%=28.912%


Widget(value='<iframe src="http://localhost:46485/index.html?ui=P_0x734463075b40_6&reconnect=auto" class="pyvi…

threshold=0.6 (manual) | solid%=25.298%


Widget(value='<iframe src="http://localhost:46485/index.html?ui=P_0x7344615dbbb0_7&reconnect=auto" class="pyvi…

threshold=0.7 (manual) | solid%=20.895%


Widget(value='<iframe src="http://localhost:46485/index.html?ui=P_0x73420cb63d00_8&reconnect=auto" class="pyvi…